In [1]:
%pip install -q setuptools
%pip install -q emlearn

In [10]:
import io
import os

from datetime import datetime, timezone

from pyodide.http import pyfetch

import numpy
import pandas
import emlearn

In [6]:

# Hostname of our IoT device
HOST = '192.168.87.142'
PORT = 80


In [5]:

async def query(host, port, resource, start_dt, end_dt, chunk_rows=600):
    """
    Call GET /query and return a 2D numpy array via numpy.load.
    start_dt / end_dt: datetime objects (UTC).
    """
    start_s = int(start_dt.replace(tzinfo=timezone.utc).timestamp())
    end_s   = int(end_dt.replace(tzinfo=timezone.utc).timestamp())

    url = 'http://{}:{}/query?resource={}&start={}&end={}&chunk_rows={}'.format(
        host, port, resource, start_s, end_s, chunk_rows)

    response = await pyfetch(url)
    data = await response.bytes()

    return numpy.load(io.BytesIO(data))

start_dt = datetime.fromisoformat('2025-06-01T05:00:00')
end_dt   = datetime.fromisoformat('2025-06-01T07:00:00')

resource = 'sensor'
arr = await query(HOST, PORT, resource, start_dt, end_dt)

arr

<class 'pyodide.http._exceptions.AbortError'>: NetworkError when attempting to fetch resource.

In [13]:


def make_noisy_xor(seed=42):
    xx, yy = numpy.meshgrid(numpy.linspace(-3, 3, 500),
                         numpy.linspace(-3, 3, 500))

    rng = numpy.random.RandomState(seed)
    X = rng.randn(300, 2)
    y = numpy.logical_xor(X[:, 0] > 0, X[:, 1] > 0)

    # Add some noise
    flip = rng.randint(300, size=15)
    y[flip] = ~y[flip]

    df = pandas.DataFrame((X * 255).astype(numpy.int16))
    df['label'] = y

    return df

def dataset_split_random(data, val_size=0.25, test_size=0.25, random_state=3, column='split'):
    """
    Split DataFrame into 3 non-overlapping parts: train,val,test with specified proportions

    Returns a new DataFrame with the rows marked by the assigned split in @column
    """
    train_size = (1.0 - val_size - test_size)
    from sklearn.model_selection import train_test_split
    
    train_val_idx, test_idx = train_test_split(data.index, test_size=test_size, random_state=random_state)
    val_ratio = (val_size / (val_size+train_size))
    train_idx, val_idx = train_test_split(train_val_idx, test_size=val_ratio, random_state=random_state)

    data = data.copy()
    data.loc[train_idx, column] = 'train'
    data.loc[val_idx, column] = 'val'
    data.loc[test_idx, column] = 'test'

    return data

dataset = make_noisy_xor()
dataset = dataset_split_random(dataset, test_size=0.10).set_index('split')

# Show colums of the data
print(dataset.head(5))


def train_model(dataset, seed=42):
    from sklearn.ensemble import RandomForestClassifier

    #feature_columns = 
    X_train = dataset.loc['train', [0, 1]]
    Y_train = dataset.loc['train', 'label']

    model = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=seed)
    model.fit(X_train, Y_train)

    return model


model = train_model(dataset)


def convert_model(model):

    model_filename = os.path.join('xor_model.csv')
    cmodel = emlearn.convert(model)
    code = cmodel.save(file=model_filename, name='xor', format='csv')

    assert os.path.exists(model_filename)
    print(f"Generated {model_filename}")

    return code

model_data =  convert_model(model)
model_data[0:100]

         0    1  label
split                 
val    126  -35   True
val    165  388  False
val    -59  -59  False
train  402  195  False
train -119  138   True
Generated xor_model.csv


'f,2\r\nc,2\r\nlb,0\r\nl,0\r\nl,1\r\nr,0\r\nr,13\r\nr,29\r\nr,44\r\nr,58\r\nr,74\r\nr,89\r\nr,102\r\nr,110\r\nr,121\r\nn,1,348.5,1,'

In [19]:
async def upload(host, port, path, content):

    url = 'http://{}:{}/files/{}'.format(host, port, path)

    body = content.encode() if isinstance(content, str) else content
    
    print('PUT', url)
    response = await pyfetch(
        url,
        method="PUT",
        body=body,
        headers={"Content-Type": "application/octet-stream"},
    )

    result = await response.json()
    return result
    
res = await upload(HOST, PORT, 'xor_model.h', model_data)
res

PUT http://192.168.87.142:80/files/xor_model.h


<class 'pyodide.http._exceptions.AbortError'>: NetworkError when attempting to fetch resource.

In [21]:
async def list(host, port):

    url = 'http://{}:{}/files'.format(host, port)
    print('GET', url)
    response = await pyfetch(
        url,
        method="GET",
        headers={"Content-Type": "application/octet-stream"},
    )

    result = await response.json()
    return result
    
res = await list(HOST, PORT)
res

GET http://192.168.87.142:80/files


<class 'pyodide.http._exceptions.AbortError'>: NetworkError when attempting to fetch resource.